# 🌌 OpenVista - Google Colab Edition

This notebook allows you to run the full OpenVista AI platform on Google Colab, leveraging a cloud GPU for image generation.

### 🛠️ Step 1: Setup Environment
Clone the repository and install all dependencies.

In [ ]:
# @title Clone and Bootstrap
import os
!git clone https://github.com/FayssalSabri/OpenVista.git
%cd OpenVista
!bash scripts/colab_bootstrap.sh

### 🔑 Step 2: Configure Ngrok
To access the web interface, you need a free Ngrok token. 
1. Get your token here: [dashboard.ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken)
2. Paste it below.

In [ ]:
# @title Ngrok Auth
NGROK_TOKEN = "" # @param {type:"string"}
os.environ["NGROK_TOKEN"] = NGROK_TOKEN

from pyngrok import ngrok
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
else:
    print("❌ Please provide an NGROK_TOKEN!")

### 🚀 Step 3: Start Services
This will start the Backend (FastAPI), the AI Worker (Celery), and the Frontend (Next.js).

In [ ]:
# @title Run OpenVista
import subprocess
import time
import nest_asyncio
from pyngrok import ngrok

# 1. Setup Tunnels
print("📡 Setting up Ngrok tunnels...")
frontend_tunnel = ngrok.connect(3000)
backend_tunnel = ngrok.connect(8000)

print(f"\n🔥 FRONTEND URL: {frontend_tunnel.public_url}")
print(f"🔗 BACKEND URL: {backend_tunnel.public_url}\n")

# 2. Update .env for Colab
with open(".env", "w") as f:
    f.write(f"REDIS_URL=redis://localhost:6379/0\n")
    f.write(f"NEXT_PUBLIC_API_URL={backend_tunnel.public_url}\n")
    f.write(f"DATA_PATH=/content/OpenVista/data\n")
    f.write(f"MOCK_WORKER=false\n")

# 3. Start Backend
print("🟢 Starting Backend...")
backend_proc = subprocess.Popen(["uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", "8000"], 
                                cwd=".", stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

# 4. Start Worker
print("🟢 Starting Worker...")
worker_proc = subprocess.Popen(["celery", "-A", "backend.worker", "worker", "--loglevel=info"], 
                               cwd=".", stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

# 5. Start Frontend
print("🟢 Starting Frontend (this might take a minute)...")
frontend_proc = subprocess.Popen(["npm", "run", "dev", "--", "-p", "3000"], 
                                 cwd="frontend", stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

print("\n✨ ALL SYSTEMS GO!")
print(f"Open this link to use OpenVista: {frontend_tunnel.public_url}")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("🛑 Stopping services...")
    backend_proc.terminate()
    worker_proc.terminate()
    frontend_proc.terminate()
    ngrok.kill()